# Descrição do Dataset: [Spaceship Titanic](https://www.kaggle.com/competitions/spaceship-titanic/data)

O dataset utilizado nesta competição é uma versão derivativa e sintética que simula um cenário de ficção científica: o objetivo é prever se um passageiro foi transportado para uma dimensão alternativa após a colisão da nave ***Spaceship Titanic*** com uma anomalia espaço-temporal.

Para construir as previsões, o conjunto de dados fornece registros pessoais recuperados do sistema de computadores danificado da nave.

## Arquivos do Projeto

* **`train.csv`**: Registros pessoais de aproximadamente dois terços (~8.700) dos passageiros, utilizados como dados de treino. Contém a variável alvo (`Transported`).
* **`test.csv`**: Registros pessoais do terço restante (~4.300) dos passageiros, utilizados como dados de teste. O objetivo é prever o valor de `Transported` para este conjunto.

---

## Dicionário de Dados (Variáveis)

O conjunto de dados possui as seguintes colunas informativas:

## Identificação e Perfil do Passageiro

* **`PassengerId`**: Um identificador único para cada passageiro no formato `gggg_pp`, onde:
* `gggg` indica o grupo com o qual o passageiro está viajando (frequentemente membros da mesma família).
* `pp` é o número do passageiro dentro daquele grupo.

* **`Name`**: O nome e sobrenome do passageiro.
* **`Age`**: A idade do passageiro.

## Detalhes da Viagem

* **`HomePlanet`**: O planeta de origem do passageiro.
* **`Destination`**: O planeta de destino onde o passageiro iria desembarcar.
* **`Cabin`**: O número da cabine onde o passageiro estava hospedado. Segue o formato `deck/num/side` (conves/número/lado), onde o lado pode ser **P** (*Port* - Bombordo) ou **S** (*Starboard* - Estibordo).
* **`CryoSleep`**: Indica se o passageiro optou por ser colocado em animação suspensa (criossono) durante a viagem. Passageiros em criossono ficam confinados em suas cabines.
* **`VIP`**: Indica se o passageiro pagou por serviços VIP especiais durante a viagem.

## Despesas a Bordo

Valor total que o passageiro gastou em cada uma das muitas comodidades de luxo da Spaceship Titanic:

* **`RoomService`**: Serviço de quarto.
* **`FoodCourt`**: Praça de alimentação.
* **`ShoppingMall`**: Shopping center.
* **`Spa`**: Centro de relaxamento/Spa.
* **`VRDeck`**: Deck de Realidade Virtual.

## Variável Alvo (*Target*)

* **`Transported`**: Indica se o passageiro foi transportado para outra dimensão (**True** ou **False**). Esta é a variável que o modelo deve prever.

# *Setup* do ambiente

Nessa seção iremos:
- Baixar as bibliotecas necessárias no projeto;
- Definir configurações globais de bibliotecas;
- Baixar o dataset.

In [ ]:
# Manipulação básica de dados
import pandas as pd

# Bibliotecas de visualização e utilitários usados nas células de EDA abaixo
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo consistente para todos os gráficos
sns.set_theme(style="whitegrid")

In [ ]:
train = pd.read_csv("./data/train.csv")
test = pd.read_csv("./data/test.csv")

print("Tabela com as primeiras linhas do conjunto de treino")

# Visualização rápida das primeiras linhas
train.head()

# Análise Exploratória dos Dados (EDA)

Nesta seção, iremos explorar a estrutura do conjunto de treino, a qualidade dos dados e os principais padrões associados ao alvo `Transported`. Antes de analisar relações e correlações, faremos uma etapa pontual de pré-processamento para separar `PassengerId` em `PassengerGroup` e `PassengerNumber` e `Cabin` em `CabinDeck`, `CabinNum` e `CabinSide`. Essa decomposição vai simplificar a leitura dos atributos e das correlações, sem transformar esses campos derivados no foco principal da EDA.


In [ ]:
# Pré-processamento pontual para simplificar a EDA
eda = train.copy()

passenger_parts = eda["PassengerId"].str.split("_", expand=True)
eda["PassengerGroup"] = passenger_parts[0]
eda["PassengerNumber"] = pd.to_numeric(passenger_parts[1], errors="coerce")

cabin_parts = eda["Cabin"].str.split("/", expand=True)
eda["CabinDeck"] = cabin_parts[0]
eda["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
eda["CabinSide"] = cabin_parts[2]

In [ ]:
# Visão geral do dataset: tipos, contagens de não nulos e memória
print("Resumo do dataset com tipos e contagem de valores não nulos")
train.info()

In [ ]:
# Descrição geral dos atributos numéricos
print("Tabela com estatísticas descritivas das variáveis numéricas")
train.describe()

In [ ]:
# Percentual de gasto zero por serviço
print("Tabela com percentual de gasto zero por serviço")
(train[["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]] == 0).mean() * 100

In [ ]:
# Função auxiliar: resumo de valores ausentes por coluna
def missing_summary(df):
    missing = df.isna().sum()
    pct = (missing / len(df) * 100).round(2)
    out = pd.DataFrame({"missing_count": missing, "missing_pct": pct})
    return out[out["missing_count"] > 0].sort_values("missing_pct", ascending=False)

In [ ]:
print("Tabela com quantidade de valores ausentes e percentual por atributo")
missing_train = missing_summary(train)
display(missing_train)

In [ ]:
print("Gráfico com quantidade de valores ausentes por atributo")
ax = missing_train["missing_count"].plot(
    kind="bar",
    figsize=(10, 4),
    title="Valores ausentes por atributo (contagem)"
)
ax.set_xlabel("Atributo")
ax.set_ylabel("Quantidade de ausentes")
ax.legend(["Quantidade"], bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

In [ ]:
print("Gráfico com percentual de valores ausentes por atributo")
ax = missing_train["missing_pct"].plot(
    kind="bar",
    figsize=(10, 4),
    title="Valores ausentes por atributo (percentual)"
)
ax.set_xlabel("Atributo")
ax.set_ylabel("Percentual (%)")
ax.legend(["Percentual"], bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distribuição do atributo alvo para checar balanceamento de classes
print("Tabela com contagem e percentual do atributo alvo")
target_counts = train["Transported"].value_counts(dropna=False)
target_pct = train["Transported"].value_counts(normalize=True, dropna=False).mul(100).round(2)
display(pd.DataFrame({"count": target_counts, "pct": target_pct}))

print("Gráfico com distribuição do atributo alvo")
ax = target_counts.plot(kind="bar", title="Distribuição do atributo Transported")
ax.set_xlabel("Transported")
ax.set_ylabel("Contagem")
plt.tight_layout()
plt.show()

In [ ]:
# Cardinalidade de campos categóricos para identificar alta/baixa variedade
print("Tabela com cardinalidade de atributos categóricos")
cat_cols = train.select_dtypes(include=["object", "str", "bool"]).columns
cardinality = train[cat_cols].nunique(dropna=False).sort_values(ascending=False)
display(cardinality.to_frame("nunique"))

In [ ]:
# Compara a taxa de transporte entre categorias principais
cat_cols = ["HomePlanet", "Destination", "CryoSleep", "VIP", "CabinDeck", "CabinSide"]

for col in cat_cols:
    tab = pd.crosstab(eda[col], eda["Transported"], normalize="index")
    tab = tab.rename(columns={False: "não_transportado", True: "transportado"})
    print(f"Tabela com taxa de transporte por categoria: {col}")
    display(tab.sort_values("transportado", ascending=False).round(3))

In [ ]:
# Gráficos para cada categoria com taxa de transporte
for col in cat_cols:
    print(f"Gráfico com taxa de transporte por categoria: {col}")
    rate = pd.crosstab(eda[col], eda["Transported"], normalize="index")
    rate = rate.rename(columns={False: "Não transportado", True: "Transportado"})
    ax = rate.plot(
        kind="bar",
        stacked=True,
        figsize=(7, 3),
        title=f"Taxa de transporte por {col}"
    )
    ax.set_xlabel(col)
    ax.set_ylabel("Proporção")
    ax.legend(title="Transported", bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
    plt.tight_layout()
    plt.show()

### Grupo do passageiro

Como `PassengerGroup` também tem alta cardinalidade, a análise abaixo foca no tamanho dos grupos formados a partir de `PassengerId`. Isso ajuda a entender se passageiros que viajam em grupo apresentam padrões diferentes de transporte, sem tratar cada grupo como uma categoria isolada.


In [ ]:
# Explora PassengerGroup pelo tamanho dos grupos
print("Tabela com distribuição dos tamanhos de PassengerGroup")
passenger_group_size = eda.groupby("PassengerGroup").size()
passenger_group_summary = (
    passenger_group_size.value_counts()
    .sort_index()
    .rename_axis("group_size")
    .to_frame("qtd_grupos")
)
passenger_group_summary["qtd_passageiros"] = passenger_group_summary.index * passenger_group_summary["qtd_grupos"]
passenger_group_summary["transportado_pct"] = (
    eda.assign(PassengerGroupSize=eda.groupby("PassengerGroup")["PassengerId"].transform("size"))
    .groupby("PassengerGroupSize")["Transported"]
    .mean()
    .mul(100)
    .round(2)
)
display(passenger_group_summary)

print("Gráfico com taxa de transporte por tamanho de PassengerGroup")
ax = passenger_group_summary["transportado_pct"].plot(
    kind="bar",
    figsize=(8, 3),
    title="Taxa de transporte por tamanho de PassengerGroup",
    color="#4C78A8",
)
ax.set_xlabel("Tamanho do PassengerGroup")
ax.set_ylabel("Transportado (%)")
plt.tight_layout()
plt.show()


## Transporte, criossono e gastos

Nas células de código abaixo, iremos separar duas ideias: gasto total mediano e presença de gasto positivo conhecido. A mediana de `TotalSpend` dos transportados irá aparecer como `0.0`, enquanto a dos não transportados irá aparecer como `907.0`; isso irá indicar que pelo menos metade dos transportados terá gasto total zero ou muito baixo, sem significar que todos não gastarão.

O que será observado:
- Percentual dos transportados que estarão em criossono.
- Percentual dos passageiros em criossono que terão algum gasto conhecido.
- Quantidade de registros com gastos ausentes, porque eles poderão esconder gasto não observado.

Resultados que serão destacados:
- Transportados em criossono: 58.12% (2483 de 4272 transportados com `CryoSleep` preenchido).
- Passageiros em criossono com algum gasto conhecido: 0.00% (0 de 3037).
- Transportados que gastarão algo: 34.38% (1505 de 4378).
- Mediana de `TotalSpend`: 907.0 para não transportados e 0.0 para transportados.
- Observação: existirão 347 passageiros em criossono com algum gasto ausente e sem gasto positivo registrado; por isso, a frase mais precisa será "nenhum gasto positivo conhecido".


In [ ]:
# Análise de transporte, criossono e gastos
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

eda = train.copy()

# Usa min_count=1 para não transformar linhas com todos os gastos ausentes em zero.
eda["TotalSpend"] = eda[spend_cols].sum(axis=1, min_count=1)
eda["SpentSomething"] = eda[spend_cols].gt(0).any(axis=1)
eda["HasMissingSpend"] = eda[spend_cols].isna().any(axis=1)

transported = eda["Transported"].eq(True)
cryosleep = eda["CryoSleep"].eq(True)
cryo_known = eda["CryoSleep"].notna()

transported_with_cryo_known = transported & cryo_known
pct_transported_in_cryo = cryosleep[transported_with_cryo_known].mean() * 100

pct_cryo_with_spend = eda.loc[cryosleep, "SpentSomething"].mean() * 100
pct_transported_with_spend = eda.loc[transported, "SpentSomething"].mean() * 100

focus = pd.DataFrame(
    [
        {
            "pergunta": "% dos transportados em criossono",
            "numerador": int((transported & cryosleep).sum()),
            "denominador": int(transported_with_cryo_known.sum()),
            "percentual": pct_transported_in_cryo,
        },
        {
            "pergunta": "% dos em criossono que gastaram algo",
            "numerador": int((cryosleep & eda["SpentSomething"]).sum()),
            "denominador": int(cryosleep.sum()),
            "percentual": pct_cryo_with_spend,
        },
        {
            "pergunta": "% dos transportados que gastaram algo",
            "numerador": int((transported & eda["SpentSomething"]).sum()),
            "denominador": int(transported.sum()),
            "percentual": pct_transported_with_spend,
        },
    ]
)

focus["percentual"] = focus["percentual"].round(2)

print("Perguntas foco")
display(focus)

print("Tabela de verificação: Transported x CryoSleep")
transport_cryo = pd.crosstab(
    eda["Transported"],
    eda["CryoSleep"],
    margins=True,
    margins_name="total",
)
display(transport_cryo)

print("Tabela de verificação: CryoSleep x gastou algo")
cryo_spend = pd.crosstab(
    eda["CryoSleep"],
    eda["SpentSomething"],
    margins=True,
    margins_name="total",
)
cryo_spend.columns = ["não_gastou", "gastou", "total"]
display(cryo_spend)

print("Registros com algum gasto ausente")
missing_spend = pd.DataFrame(
    {
        "grupo": ["transportados", "em_criossono", "em_criossono_sem_gasto_positivo"],
        "registros_com_gasto_ausente": [
            int((transported & eda["HasMissingSpend"]).sum()),
            int((cryosleep & eda["HasMissingSpend"]).sum()),
            int((cryosleep & ~eda["SpentSomething"] & eda["HasMissingSpend"]).sum()),
        ],
    }
)
display(missing_spend)


### Mediana de gasto total por transporte

Na próxima célula de código, o gráfico irá materializar a mediana citada acima. A barra dos transportados irá ficar em `0`, indicando que pelo menos metade desse grupo terá gasto total zero registrado; a tabela seguinte irá conectar esse padrão ao `CryoSleep`.


In [ ]:
# Gráfico que irá ilustrar a mediana de gasto total por alvo
median_total_spend = (
    eda.groupby("Transported")["TotalSpend"]
    .median()
    .rename(index={False: "Não transportado", True: "Transportado"})
)

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(
    median_total_spend.index,
    median_total_spend.values,
    color=["#4C78A8", "#F58518"],
)

ax.set_title("Mediana de gasto total por Transported")
ax.set_xlabel("Transported")
ax.set_ylabel("Mediana de TotalSpend")
ax.bar_label(
    bars,
    labels=[f"{value:.0f}" for value in median_total_spend.values],
    padding=3,
)
ax.set_ylim(0, max(median_total_spend.max() * 1.15, 1))

plt.tight_layout()
plt.show()


In [ ]:
# Tabela que irá relacionar gasto total com criossono
total_spend_by_cryo = (
    eda.groupby("CryoSleep", dropna=False)
    .agg(
        qtd_passageiros=("PassengerId", "size"),
        mediana_total_spend=("TotalSpend", "median"),
        média_total_spend=("TotalSpend", "mean"),
        pct_com_gasto=("SpentSomething", "mean"),
    )
    .reset_index()
)

total_spend_by_cryo["pct_com_gasto"] = (total_spend_by_cryo["pct_com_gasto"] * 100).round(2)
total_spend_by_cryo[["mediana_total_spend", "média_total_spend"]] = total_spend_by_cryo[
    ["mediana_total_spend", "média_total_spend"]
].round(2)

display(total_spend_by_cryo)


## Correlações numéricas

Nas células de código abaixo, é calculada a correlação entre variáveis numéricas, incluindo o alvo `Transported` convertido para 0/1. A tabela ordena as variáveis pela intensidade absoluta da relação linear com o alvo, e o heatmap mostra também a relação entre as próprias variáveis numéricas.

O que será observado:
- Variáveis numéricas com maior correlação linear, em módulo, com `Transported`.
- Pares de variáveis muito correlacionadas entre si, que podem indicar redundância.
- Sinal da correlação: valores positivos e negativos indicam direções diferentes da relação linear.

Resultados que se destacam:
- `RoomService`, `Spa`, `VRDeck` e `TotalSpend` aparecem com as correlações negativas mais fortes em módulo, embora ainda moderadas/fracas.
- `Age`, `FoodCourt` e `ShoppingMall` aparecem com correlações mais fracas com `Transported`.
- `TotalSpend` tem correlação alta com algumas colunas de gasto, o que é esperado por ser uma soma dessas despesas.


In [ ]:
# Análise de correlação das variáveis numéricas com o alvo
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]

eda_num = train.copy()
eda_num["TotalSpend"] = eda_num[spend_cols].sum(axis=1, min_count=1)
eda_num["Transported"] = eda_num["Transported"].astype(int)

num_cols = eda_num.select_dtypes(include="number").columns
corr = eda_num[num_cols].corr()

print("Tabela com correlação das variáveis numéricas com o alvo")
# Ordena a correlação com o alvo pela intensidade absoluta.
target_corr = corr["Transported"].drop("Transported")
target_corr = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index)
display(target_corr)

print("Gráfico heatmap com correlações entre variáveis numéricas")
# Heatmap das relações numéricas
plt.figure(figsize=(7, 5))
sns.heatmap(corr, cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Heatmap de correlações numéricas")
plt.tight_layout()
plt.show()


## Síntese da EDA

A partir das tabelas e gráficos da EDA, os principais aprendizados são:

- O conjunto de treino tem 8693 registros e 14 colunas.
- O alvo `Transported` está praticamente balanceado: 50.36% transportados e 49.64% não transportados.
- Os valores ausentes são baixos e distribuídos entre várias colunas; a maior ausência está em `CryoSleep`, com 217 registros (2.50%).
- As colunas de gastos têm muitos zeros: entre 61.24% e 64.27% dos registros têm gasto zero em cada serviço individual.
- A idade tem mediana de 27 anos, com grupos transportados e não transportados muito próximos em idade mediana (26.0 vs. 27.0).
- `HomePlanet` apresenta uma diferença importante: Europa tem a maior taxa de transporte (65.88%), seguida por Mars (52.30%) e Earth (42.39%).
- `Destination` também mostra diferença: 55 Cancri e tem a maior taxa de transporte (61.00%), seguida por PSO J318.5-22 (50.38%) e TRAPPIST-1e (47.12%).
- `CryoSleep` é um dos sinais mais fortes da EDA: passageiros em criossono têm taxa de transporte de 81.76%, enquanto passageiros fora do criossono têm 32.89%.
- Passageiros VIP têm taxa de transporte menor (38.19%) do que passageiros não VIP (50.63%).
- `PassengerGroup` mostra que a maioria dos grupos tem apenas 1 passageiro (4805 grupos), mas grupos maiores, sobretudo de 2 a 7 passageiros, apresentam taxas de transporte mais altas do que grupos unitários; o pico aparece em grupos de 4 passageiros (64.08%), enquanto grupos de 8 passageiros caem para 39.42% e são raros.
- Em `CabinDeck`, os decks B e C têm as maiores taxas de transporte: B com 73.43% e C com 68.01%. O deck T tem taxa baixa (20.00%), mas tem apenas 5 registros, então esse resultado pede cautela.
- Em `CabinSide`, o lado S tem maior taxa de transporte (55.50%) do que o lado P (45.13%).
- A mediana de `TotalSpend` é 907.0 para não transportados e 0.0 para transportados. Isso não significa que nenhum transportado gaste algo; 1505 transportados têm algum gasto positivo conhecido (34.38%).
- A relação entre gasto e criossono fica bem clara: passageiros em `CryoSleep=True` têm mediana de `TotalSpend` igual a 0.0 e 0.00% com gasto positivo conhecido; passageiros em `CryoSleep=False` têm mediana de 1019.0 e 90.48% com algum gasto.
- Entre os transportados com `CryoSleep` preenchido, 58.12% estão em criossono.
- As correlações numéricas com `Transported` são fracas a moderadas: `RoomService` (-0.245), `Spa` (-0.221), `VRDeck` (-0.207) e `TotalSpend` (-0.200) são as mais fortes em módulo, enquanto `Age` (-0.075), `FoodCourt` (0.047) e `ShoppingMall` (0.010) são muito fracas.


# Pré-processamento